In [0]:
%sql

SELECT
    order_status,
    payment_status,
    COUNT(*) AS total_applications,
    SUM(COALESCE(brokerage_inr, 0)) AS total_brokerage,
    COUNT_IF(new_payment_success_flag = TRUE) AS payment_success_count,
    COUNT_IF(allotment_doneflag = TRUE) AS allotted_count

FROM ipo_orders

GROUP BY order_status, payment_status

ORDER BY total_applications DESC;

In [0]:
%sql

SELECT
    third_party_id,
    groww_order_id,
    order_status,
    payment_status,
    new_payment_success_flag,
    allotment_doneflag,
    brokerage_inr,
    upi_amount_blocked,
    mandate_status,
    payment_success_time,
    approved_time,
    not_alloted_time

FROM ipo_orders

WHERE
    (order_status = 'NOT_ALLOTED'
        AND payment_status = 'PENDING')

    OR

    (order_status = 'APPROVED'
        AND payment_status = 'SUCCESS')

LIMIT 20;

Look at these two anomalies:
- 885 NOT_ALLOTED + PENDING applications have ₹1 brokerage despite unsuccessful payments.
- 1 NOT_ALLOTED + FAILED application also has ₹1 brokerage.
Conversely, 191 APPROVED + SUCCESS applications have zero brokerage despite successful payments and exchange approval.

Therefore, the supplied brokerage_inr column is inconsistent with the stated business rule.

We should calculate revenue ourselves rather than blindly summing that column.
2. Our final IPO revenue definition

In [0]:
%sql

CREATE OR REPLACE TABLE ipo_successful AS

SELECT
    third_party_id,
    groww_order_id,
    exchange_order_id,
    created_at,
    isin,
    symbol,
    category,
    new_payment_success_flag,
    allotment_doneflag,
    upi_amount_blocked,
    shares_allocated,
    name,
    max_bid_value,
    is_sme,
    os_name,
    order_status,
    payment_status,
    mandate_status,
    mandate_set,
    approve_click,
    decline_click,
    mandate_discovered,
    discovery_source,

    1 AS ipo_revenue

FROM ipo_orders

WHERE
    order_status IN (
        'ALLOTMENT_DONE',
        'NOT_ALLOTED',
        'APPROVED'
    )
    AND new_payment_success_flag = TRUE;

In [0]:
%sql

CREATE OR REPLACE TABLE ipo_user_metrics AS

SELECT

    third_party_id,

    -- Revenue
    SUM(ipo_revenue) AS total_brokerage,

    -- Application activity
    COUNT(DISTINCT groww_order_id)
        AS total_applications,

    COUNT(DISTINCT TO_DATE(created_at))
        AS application_days,

    -- IPO diversification
    COUNT(DISTINCT symbol)
        AS distinct_ipos,

    COUNT(DISTINCT category)
        AS distinct_categories,

    -- Allotment behaviour
    SUM(CASE
        WHEN allotment_doneflag = TRUE
        THEN 1 ELSE 0
    END) AS allotted_applications,

    -- Payment behaviour
    SUM(CASE
        WHEN new_payment_success_flag = TRUE
        THEN 1 ELSE 0
    END) AS payment_success_count,

    -- Mandate behaviour
    SUM(CASE
        WHEN mandate_set = 1
        THEN 1 ELSE 0
    END) AS mandates_set,

    SUM(CASE
        WHEN mandate_discovered = 1
        THEN 1 ELSE 0
    END) AS existing_mandates_used,

    SUM(COALESCE(shares_allocated, 0))
        AS total_shares_allocated,

    SUM(COALESCE(upi_amount_blocked, 0))
        AS total_amount_blocked,

    MAX(created_at) AS last_application_date

FROM ipo_successful

GROUP BY third_party_id;

In [0]:
%sql

CREATE OR REPLACE TABLE ipo_user_segments AS

SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment

FROM ipo_user_metrics;

In [0]:
%sql

SELECT

    value_segment,

    COUNT(*) AS total_users,

    SUM(total_applications) AS total_applications,

    SUM(total_brokerage) AS total_revenue,

    ROUND(
        100.0 * SUM(total_brokerage) /
        NULLIF(SUM(SUM(total_brokerage)) OVER (), 0),
        2
    ) AS revenue_contribution_pct

FROM ipo_user_segments

GROUP BY value_segment

ORDER BY value_segment;

In [0]:
%sql

SELECT

    CASE
        WHEN value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    COUNT(*) AS total_users,

    ROUND(AVG(total_applications), 2)
        AS avg_applications,

    ROUND(AVG(application_days), 2)
        AS avg_application_days,

    ROUND(AVG(distinct_ipos), 2)
        AS avg_distinct_ipos,

    ROUND(AVG(allotted_applications), 2)
        AS avg_allotments,

    ROUND(AVG(total_amount_blocked), 2)
        AS avg_amount_blocked,

    ROUND(AVG(mandates_set), 2)
        AS avg_mandates_set

FROM ipo_user_segments

GROUP BY 1;

Analyse IPO category preferences
We want to understand whether high-value users participate more in retail, shareholder or employee categories.

In [0]:
%sql

SELECT

    CASE
        WHEN s.value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    o.category,

    COUNT(DISTINCT o.third_party_id) AS users,

    COUNT(DISTINCT o.groww_order_id) AS applications,

    SUM(o.ipo_revenue) AS revenue

FROM ipo_successful o

JOIN ipo_user_segments s
    ON o.third_party_id = s.third_party_id

GROUP BY 1, 2

ORDER BY 1, 2;